In [2]:
!pip install transformers datasets scikit-learn tqdm matplotlib seaborn torch

import os
import re
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, BertModel 
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


if not os.path.exists('bert_checkpoints/plots'):
    os.makedirs('bert_checkpoints/plots')

print(f"Zariadenie: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.3/532.3 MB 29.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 36.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 68.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 67.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 130.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.5/201.5 MB 71.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 212.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 115.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 236.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 52.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
df = pd.read_csv('labeled_data.csv')

def clean_text(text):
    text = text.lower()
    text = re.sub(r'@[\w\-]+', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'\brt\b', '', text)
    text = re.sub(r'&amp;|&#\d+;', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)
df = df.drop_duplicates(subset=['clean_tweet'])
df = df[df['clean_tweet'] != ''].dropna()

X = df['clean_tweet']
y = df['class']

In [5]:
class SimpleDataset(Dataset):
    
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts.values
        self.labels = labels.values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, 
            padding='max_length', 
            truncation=True, 
            max_length=64, 
            return_tensors='pt'
        )
        return {
            'ids': encoding['input_ids'].flatten(),
            'mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [6]:
class BertClassifier(nn.Module):
    
    def __init__(self, num_classes=3):
        super(BertClassifier, self).__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, ids, mask):
        output = self.bert(input_ids=ids, attention_mask=mask)
        pooled_output = output.pooler_output
        out = self.dropout(pooled_output)
        return self.classifier(out)

In [7]:
def save_fold_loss_curve(history, fold_num):
    
    plt.figure(figsize=(4, 2.5)) 
    plt.plot(history['train'], label='Train Loss', linewidth=1.5, marker='o', markersize=4)
    plt.plot(history['val'], label='Val Loss', linewidth=1.5, marker='s', markersize=4)
    plt.title(f'BERT Loss: Fold {fold_num}', fontsize=9)
    plt.xlabel('Epocha', fontsize=8)
    plt.ylabel('Loss', fontsize=8)
    plt.xticks(range(len(history['train'])), range(1, len(history['train']) + 1), fontsize=7)
    plt.yticks(fontsize=7)
    plt.legend(fontsize=7)
    plt.grid(True, linestyle='--', alpha=0.5)
    
    plot_path = f'bert_checkpoints/plots/fold_{fold_num}_loss_curves.png'
    plt.savefig(plot_path, dpi=300, bbox_inches='tight')
    plt.close()

In [8]:
def train_model_advanced(model, train_loader, val_loader, fold, epochs=5, patience=2):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)
    criterion = torch.nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    epochs_no_improve = 0  
    fold_history = {'train': [], 'val': []}

    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        train_loop = tqdm(train_loader, desc=f"BERT Fold {fold} | Ep {epoch+1}")
        
        for batch in train_loop:
            optimizer.zero_grad()
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(ids, mask)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_train_loss += loss.item()

        model.eval()
        total_val_loss = 0
        correct = 0
        with torch.no_grad():
            for batch in val_loader:
                ids = batch['ids'].to(device)
                mask = batch['mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(ids, mask)
                loss = criterion(outputs, labels)
                total_val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                correct += (preds == labels).sum().item()

        avg_train = total_train_loss / len(train_loader)
        avg_val = total_val_loss / len(val_loader)
        fold_history['train'].append(avg_train)
        fold_history['val'].append(avg_val)

        print(f"Ep {epoch+1}: Train Loss {avg_train:.4f} | Val Loss {avg_val:.4f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), f"bert_checkpoints/best_bert_fold_{fold}.pt")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience: break
    return fold_history

In [9]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

global_best_loss = float('inf')
best_val_indices = None

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- BERT FOLD {fold+1} ---")
    train_loader = DataLoader(SimpleDataset(X.iloc[train_idx], y.iloc[train_idx], tokenizer), batch_size=16, shuffle=True)
    val_loader = DataLoader(SimpleDataset(X.iloc[val_idx], y.iloc[val_idx], tokenizer), batch_size=16)
    
    model = BertClassifier().to(device)
    history = train_model_advanced(model, train_loader, val_loader, fold=fold+1)
    save_fold_loss_curve(history, fold_num=fold+1)
    
 
    if history['val'][-1] < global_best_loss:
        global_best_loss = history['val'][-1]
        best_val_indices = val_idx
        torch.save(model.state_dict(), "bert_checkpoints/absolute_best_bert_model.pt")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]


--- BERT FOLD 1 ---


...

Ep 1: Train Loss 0.3031 | Val Loss 0.2382


BERT Fold 5 | Ep 2:   0%|          | 0/1223 [00:00<?, ?it/s]

Ep 2: Train Loss 0.2117 | Val Loss 0.2523


BERT Fold 5 | Ep 3:   0%|          | 0/1223 [00:00<?, ?it/s]

Ep 3: Train Loss 0.1640 | Val Loss 0.2547


In [12]:
def get_detailed_report(model, loader, title="VÝSLEDOK"):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            logits = model(batch['ids'].to(device), batch['mask'].to(device))
            all_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            all_labels.extend(batch['label'].numpy())
    
    print(f"\n=== {title} ===")
    print(classification_report(all_labels, all_preds, target_names=['Hate', 'Offensive', 'Neither'], digits=4))
    return all_labels, all_preds


final_bert = BertClassifier().to(device)
final_bert.load_state_dict(torch.load("bert_checkpoints/absolute_best_bert_model.pt"))
val_loader_final = DataLoader(SimpleDataset(X.iloc[best_val_indices], y.iloc[best_val_indices], tokenizer), batch_size=16)

y_true, y_pred = get_detailed_report(final_bert, val_loader_final, "FINÁLNY REPORT: BERT")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



=== FINÁLNY REPORT: BERT ===
              precision    recall  f1-score   support

        Hate     0.5208    0.2660    0.3521       282
   Offensive     0.9325    0.9702    0.9510      3787
     Neither     0.9043    0.8878    0.8960       820

    accuracy                         0.9157      4889
   macro avg     0.7859    0.7080    0.7330      4889
weighted avg     0.9040    0.9157    0.9072      4889

